# <span style="color: teal;">Extracting Nearest Pixels from Raster Data Based on Station Coordinates</span>

This script processes raster data files and extracts the nearest pixel values for each station based on the station's coordinates. The search is done within a defined radius using Xarray to handle raster data efficiently. Results are saved to CSV files for each raster dataset, containing the corresponding nearest pixel values for each station.

### <span style="color: teal;">Libraries</span>

In [32]:
import os
import pandas as pd
import rioxarray
import rasterio
import numpy as np
from tqdm import tqdm
from datetime import datetime

### <span style="color: teal;">Load Station Data</span>

In [25]:
# Load station data from a CSV file and extract relevant columns
def load_station_data(file_path):
    df_daily_precip = pd.read_csv(file_path)
    stations_coords_df = df_daily_precip[['Station Name', 'LAT', 'LON', 'ALTITUDE']].drop_duplicates().reset_index(drop=True)
    return stations_coords_df

### <span style="color: teal;">Define Raster Folder Paths </span>

In [26]:
# Define the paths to the raster datasets
def define_raster_folders():
    raster_folders = {
        "CWP": "Data_original/CWP_merged",
        "CER": "Data_original/CER_merged",
        "COT": "Data_original/COT_merged",
        "Wind": "Data_original/wind",
        "ASPECT": "Data_original/static/ASPECT_greece.tif",
        "SLOPE": "Data_original/static/SLOPE_greece.tif",
        "TWI": "Data_original/static/TWI.tif",
        "TPI": "Data_original/static/TPI.tif",
        "srtmMtpiGreece": "Data_original/static/srtmMtpiGreece.tif",
        "landformsGreece": "Data_original/static/landformsGreece.tif", 
        "topoDiversity": "Data_original/static/topoDiversity.tif",
        "NDVI": "Data_original/ndvi",
        "EVI": "Data_original/evi",
        "LST": "Data_original/LST",
        "Water_Vapor_Column": "Data_original/Water_Vapor_Column"
    }
    return raster_folders

### <span style="color: teal;">Function to Calculate Pixel Size from a Raster File </span>

In [ ]:
def calculate_pixel_size(tif_path):
    with rasterio.open(tif_path) as raster:
        pixelSizeX, pixelSizeY = raster.res
    return pixelSizeX, pixelSizeY

### <span style="color: teal;">Function to Find All .tif Files in a Directory </span>

In [1]:
def find_tif_files(directory):
    tif_files = []
    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".tif"):
                tif_files.append(os.path.join(root, file))
    return tif_files

### <span style="color: teal;">Function to Extract Date from Filename </span>

In [2]:
# Function to extract date from filename
def extract_date_from_filename(filename):
    # For CER, COT, CWP files (cropped_modis_MOD06_L2.A2010001.0915.061.2017308135353_HEGOUT.tif)
    if "MOD06" in filename:
        date_str = re.search(r"A(\d{7})", filename).group(1)
        date = datetime.strptime(date_str, "%Y%j").strftime("%Y-%m-%d")  # Convert Julian date to YYYY-MM-DD
    # For IMERG files (e.g., 3B-HHR-L.MS.MRG.3IMERG.20010101-S023000-E025959.0150.V07B.3hr.tif)
    elif "3IMERG" in filename:
        date_str = re.search(r"3IMERG\.(\d{8})", filename).group(1)
        date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    # For wind files (20100101_wind.tif)
    elif "wind" in filename:
        date_str = re.search(r"(\d{8})_wind", filename).group(1)
        date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    else:
        date = None  # Handle other cases if needed
    return date

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


### <span style="color: teal;">Function to Process Raster Files and Extract Values Based on Station Coordinates </span>

In [3]:
# Function to process raster files and extract values based on station coordinates
def process_raster_files(dataset_name, raster_files, dataset_folder, stations_coords_df, raster_folders):
    results = []
    
    for tif_file in tqdm(raster_files, desc=f"Processing {dataset_name}", unit="raster"):
        tif_path = tif_file  # Full path directly
        
        try:
            # Calculate pixel size for this raster
            pixelSizeX, pixelSizeY = calculate_pixel_size(tif_path)
            pixel_size = max(pixelSizeX, pixelSizeY)  # The larger value as the pixel size
            search_radius = pixel_size * 2  # Define search radius based on pixel size

            ds = rioxarray.open_rasterio(tif_path, masked=True)
        except Exception as e:
            print(f"Error opening {tif_file}: {e}")
            continue

        # Extract date from the filename
        date = extract_date_from_filename(os.path.basename(tif_file))

        for _, row in stations_coords_df.iterrows():
            station_name, lat, lon, altitude = row[['Station Name', 'LAT', 'LON', 'ALTITUDE']]
            
            try:
                # Handle multi-band rasters
                if len(ds.band) > 1:
                    band_values = {}
                    for band in range(1, len(ds.band) + 1):
                        # Find the nearest point within the search radius
                        nearest_point = ds.sel(
                            x=lon, y=lat, band=band, method="nearest", tolerance=search_radius
                        )
                        
                        # Check if the nearest point is within the search radius
                        if (
                            abs(nearest_point.x - lon) <= search_radius and
                            abs(nearest_point.y - lat) <= search_radius
                        ):
                            value = nearest_point.values.item()
                        else:
                            value = None  # Outside the search radius
                        
                        if np.isnan(value) or value <= 0:
                            value = None  
                        
                        band_values[f"Band {band}"] = value
                    
                    results.append({
                        "Dataset": dataset_name,
                        "Tif Name": os.path.basename(tif_file),
                        "Date": date,  
                        "Station Name": station_name,
                        "Station LAT": lat,
                        "Station LON": lon,
                        "Altitude": altitude,
                        **band_values  
                    })
                else:
                    # Handle single-band rasters
                    # Find the nearest point within the search radius
                    nearest_point = ds.sel(
                        x=lon, y=lat, method="nearest", tolerance=search_radius
                    )
                    
                    # Check if the nearest point is within the search radius
                    if (
                        abs(nearest_point.x - lon) <= search_radius and
                        abs(nearest_point.y - lat) <= search_radius
                    ):
                        value = nearest_point.values.item()
                    else:
                        value = None  # Outside the search radius
                    
                    if np.isnan(value) or value <= 0:
                        value = None 
                    
                    results.append({
                        "Dataset": dataset_name,
                        "Tif Name": os.path.basename(tif_file),
                        "Date": date,  
                        "Station Name": station_name,
                        "Station LAT": lat,
                        "Station LON": lon,
                        f"Value {dataset_name}": value,
                        "Altitude": altitude
                    })
            except Exception as e:
                print(f"Error processing {station_name} in {tif_file}: {e}")

    df = pd.DataFrame(results)
    output_file = os.path.join(dataset_folder, f"nearest_values_{dataset_name}.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved results to {output_file}")

### <span style="color: teal;">Main Workflow Function </span>

In [39]:
# Main workflow function to load station data, define raster folders, and process raster files for each dataset
def main():
    # Load station data
    stations_coords_df = load_station_data('df_daily_precip.csv')

    # Define raster folders
    raster_folders = define_raster_folders()

    # Define output directory
    main_output_dir = "Nearest_pixels_to_stations4"
    os.makedirs(main_output_dir, exist_ok=True)

    # Process each dataset
    for dataset_name, rasters_folder_path in raster_folders.items():
        dataset_folder = os.path.join(main_output_dir, dataset_name)
        os.makedirs(dataset_folder, exist_ok=True)

        # Handle Water Vapor Column separately due to nested folder structure
        if dataset_name == "Water_Vapor_Column":
            raster_files = find_tif_files(rasters_folder_path)
        else:
            # Check if the path is a directory or a single file
            if os.path.isdir(rasters_folder_path):
                raster_files = [f for f in os.listdir(rasters_folder_path) if f.endswith(".tif")]
                raster_files = [os.path.join(rasters_folder_path, f) for f in raster_files]
            else:
                raster_files = [rasters_folder_path]  # Single file case

        if raster_files:
            process_raster_files(dataset_name, raster_files, dataset_folder, stations_coords_df, raster_folders)

# Run the script
main()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


Processing CWP: 100%|█████████████████████████████████████████████████| 10130/10130 [07:35<00:00, 22.25raster/s]


Saved results to Nearest_pixels_to_stations4/CWP/nearest_values_CWP.csv


Processing CER: 100%|█████████████████████████████████████████████████| 10130/10130 [06:43<00:00, 25.12raster/s]


Saved results to Nearest_pixels_to_stations4/CER/nearest_values_CER.csv


...

Saved results to Nearest_pixels_to_stations4/EVI/nearest_values_EVI.csv


Processing LST: 100%|█████████████████████████████████████████████████| 10744/10744 [07:29<00:00, 23.90raster/s]


Saved results to Nearest_pixels_to_stations4/LST/nearest_values_LST.csv


Processing Water_Vapor_Column: 100%|████████████████████████████████████| 4807/4807 [03:26<00:00, 23.22raster/s]


Saved results to Nearest_pixels_to_stations4/Water_Vapor_Column/nearest_values_Water_Vapor_Column.csv


## <span style="color: #FFA500;">Nearest pixels only for wind</span>

In [38]:
import os
import re
from datetime import datetime
import rasterio
import rioxarray
import numpy as np
import pandas as pd
from tqdm import tqdm

# Load station data from a CSV file and extract relevant columns
def load_station_data(file_path):
    df_daily_precip = pd.read_csv(file_path)
    stations_coords_df = df_daily_precip[['Station Name', 'LAT', 'LON', 'ALTITUDE']].drop_duplicates().reset_index(drop=True)
    return stations_coords_df

# Function to calculate pixel size from a raster file using rasterio
def calculate_pixel_size(tif_path):
    with rasterio.open(tif_path) as raster:
        pixelSizeX, pixelSizeY = raster.res
    return pixelSizeX, pixelSizeY

# Function to extract date from filename (for wind files)
def extract_date_from_filename(filename):
    # For wind files (20100101_wind.tif)
    if "wind" in filename:
        date_str = re.search(r"(\d{8})_wind", filename).group(1)
        date = datetime.strptime(date_str, "%Y%m%d").strftime("%Y-%m-%d")
    else:
        date = None  # Handle other cases if needed
    return date

# Function to process wind raster files and extract values based on station coordinates
def process_wind_raster_files(wind_raster_files, stations_coords_df, output_dir):
    results = []
    
    for tif_file in tqdm(wind_raster_files, desc="Processing Wind", unit="raster", ncols=100):  
        tif_path = tif_file  # Use the full path directly
        
        try:
            # Calculate pixel size for this raster
            pixelSizeX, pixelSizeY = calculate_pixel_size(tif_path)
            pixel_size = max(pixelSizeX, pixelSizeY)  # The larger value as the pixel size
            search_radius = pixel_size * 2  # Define search radius based on pixel size

            # Open the multi-band raster file
            ds = rioxarray.open_rasterio(tif_path, masked=True)
        except Exception as e:
            print(f"Error opening {tif_file}: {e}")
            continue

        # Extract date from the filename
        date = extract_date_from_filename(os.path.basename(tif_file))

        for _, row in stations_coords_df.iterrows():
            station_name, lat, lon, altitude = row[['Station Name', 'LAT', 'LON', 'ALTITUDE']]

            try:
                # Initialize a dictionary to store band values
                band_values = {}

                # Loop through all bands (4 bands for wind TIFFs)
                for band in range(1, len(ds.band) + 1):
                    # Find the nearest point within the search radius for the current band
                    nearest_point = ds.sel(
                        x=lon, y=lat, band=band, method="nearest", tolerance=search_radius
                    )
                    
                    # Check if the nearest point is within the search radius
                    if (
                        abs(nearest_point.x - lon) <= search_radius and
                        abs(nearest_point.y - lat) <= search_radius
                    ):
                        # If the result is an array, we need to access the scalar value.
                        if nearest_point.values.ndim == 0:
                            value = nearest_point.values.item()  # If it's already a scalar
                        else:
                            value = nearest_point.values[0]  # Access the scalar from the array
                    else:
                        value = None  # Outside the search radius
                    
                    if np.isnan(value) or value <= 0:
                        value = None  

                    # Store the value for the current band
                    band_values[f"Band {band}"] = value

                # Append the results for all bands
                results.append({
                    "Dataset": "Wind",
                    "Tif Name": os.path.basename(tif_file),
                    "Date": date,  
                    "Station Name": station_name,
                    "Station LAT": lat,
                    "Station LON": lon,
                    "Altitude": altitude,
                    **band_values  
                })
            except Exception as e:
                print(f"Error processing {station_name} in {tif_file}: {e}")

    # Create DataFrame and save results to CSV
    df = pd.DataFrame(results)

    # Define output directory for results
    os.makedirs(output_dir, exist_ok=True)

    # Save results to CSV in the Nearest_pixels_to_stations3 directory
    output_file = os.path.join(output_dir, "nearest_values_wind.csv")
    df.to_csv(output_file, index=False)
    print(f"Saved results to {output_file}")

# Main function to load station data and process the wind raster files
def main():
    # Load station data
    stations_coords_df = load_station_data('df_daily_precip.csv')

    # Define wind raster folder path
    wind_rasters_folder = "Data_original/wind"
    
    # Find wind raster files
    wind_raster_files = [f for f in os.listdir(wind_rasters_folder) if f.endswith(".tif")]
    wind_raster_files = [os.path.join(wind_rasters_folder, f) for f in wind_raster_files]

    # Define the output directory
    output_dir = "Nearest_pixels_to_stations3"
    
    # Process the wind raster files
    if wind_raster_files:
        process_wind_raster_files(wind_raster_files, stations_coords_df, output_dir)

# Run the script
main()

Processing Wind: 100%|██████████████████████████████████████| 5112/5112 [08:10<00:00, 10.42raster/s]


Saved results to Nearest_pixels_to_stations3/nearest_values_wind.csv
